<a href="https://colab.research.google.com/github/sabinereis28/projeto-gestao-logexpress-2026/blob/main/05-pesquisa-operacional/2_otimizacao_pesquisa_operacional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pulp
import pulp
print("Biblioteca PuLP instalada com sucesso!")

Biblioteca PuLP instalada com sucesso!


In [2]:
from pulp import LpMaximize, LpProblem, LpVariable, LpStatus

# ----------------------------------------------------
# CENÁRIO BASE
# ----------------------------------------------------
# x_A: Quantidade de Fretes Locais (Curta distância)
# x_B: Quantidade de Fretes Estaduais (Longa distância)
model_max = LpProblem(name="Maximização_Lucro_LogExpress", sense=LpMaximize)

xA = LpVariable(name="Fretes_Locais", lowBound=0, cat="Integer")
xB = LpVariable(name="Fretes_Estaduais", lowBound=0, cat="Integer")

# Função Objetivo: Lucro líquido por frete (R$ 300 Local vs R$ 800 Estadual)
model_max += 300 * xA + 800 * xB, "Lucro_Total"

# Restrições de Recursos da Empresa
model_max += (2 * xA + 6 * xB <= 160, "Horas_Motoristas")  # Limite de 160 horas de jornada
model_max += (15 * xA + 50 * xB <= 1200, "Combustivel_Litros") # Limite de 1200L de combustível

model_max.solve()

print("=== PROBLEMA 1: CENÁRIO BASE ===")
print(f"Status: {LpStatus[model_max.status]}")
print(f"Fretes Locais (Tipo A): {xA.varValue}")
print(f"Fretes Estaduais (Tipo B): {xB.varValue}")
print(f"Lucro Máximo Obtido: R$ {model_max.objective.value():,.2f}\n")

# ----------------------------------------------------
# CENÁRIO WHAT-IF (Cenário de Crise: Escassez de Combustível)
# ----------------------------------------------------
# Suponha uma greve ou alta no combustível que reduz o limite de 1200L para 500L
model_max_crise = LpProblem(name="Maximização_Crise_Combustivel", sense=LpMaximize)

xA_c = LpVariable(name="Fretes_Locais", lowBound=0, cat="Integer")
xB_c = LpVariable(name="Fretes_Estaduais", lowBound=0, cat="Integer")

model_max_crise += 300 * xA_c + 800 * xB_c, "Lucro_Total"
model_max_crise += (2 * xA_c + 6 * xB_c <= 160, "Horas_Motoristas")
model_max_crise += (15 * xA_c + 50 * xB_c <= 500, "Combustivel_Restrito") # Redução para 500L

model_max_crise.solve()

print("=== PROBLEMA 1: CENÁRIO WHAT-IF (CRISE DE COMBUSTÍVEL) ===")
print(f"Status: {LpStatus[model_max_crise.status]}")
print(f"Fretes Locais (Tipo A): {xA_c.varValue}")
print(f"Fretes Estaduais (Tipo B): {xB_c.varValue}")
print(f"Lucro Reduzido na Crise: R$ {model_max_crise.objective.value():,.2f}")

=== PROBLEMA 1: CENÁRIO BASE ===
Status: Optimal
Fretes Locais (Tipo A): 80.0
Fretes Estaduais (Tipo B): 0.0
Lucro Máximo Obtido: R$ 24,000.00

=== PROBLEMA 1: CENÁRIO WHAT-IF (CRISE DE COMBUSTÍVEL) ===
Status: Optimal
Fretes Locais (Tipo A): 33.0
Fretes Estaduais (Tipo B): 0.0
Lucro Reduzido na Crise: R$ 9,900.00


In [3]:
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, LpStatus

# ----------------------------------------------------
# CENÁRIO BASE (Modelo de Transporte)
# ----------------------------------------------------
sedes = ['Galpao_SP', 'Galpao_RJ']
clientes = ['Cliente_C1', 'Cliente_C2', 'Cliente_C3']

# Capacidade das Sedes (Oferta)
oferta = {'Galpao_SP': 50, 'Galpao_RJ': 40}

# Demanda dos Clientes
demanda = {'Cliente_C1': 30, 'Cliente_C2': 35, 'Cliente_C3': 25}

# Custos de Transporte por viagem (R$)
custos = {
    'Galpao_SP': {'Cliente_C1': 100, 'Cliente_C2': 150, 'Cliente_C3': 200},
    'Galpao_RJ': {'Cliente_C1': 180, 'Cliente_C2': 120, 'Cliente_C3': 110}
}

model_min = LpProblem(name="Minimização_Custos_Transporte", sense=LpMinimize)

# Variáveis de Decisão x_ij
rotas = LpVariable.dicts("Rota", (sedes, clientes), lowBound=0, cat="Integer")

# Função Objetivo
model_min += lpSum([rotas[i][j] * custos[i][j] for i in sedes for j in clientes]), "Custo_Total_Transporte"

# Restrições de Oferta (Não enviar mais do que a capacidade)
for i in sedes:
    model_min += lpSum([rotas[i][j] for j in clientes]) <= oferta[i], f"Oferta_{i}"

# Restrições de Demanda (Atender a exata necessidade do cliente)
for j in clientes:
    model_min += lpSum([rotas[i][j] for i in sedes]) == demanda[j], f"Demanda_{j}"

model_min.solve()

print("=== PROBLEMA 2: CENÁRIO BASE ===")
print(f"Status: {LpStatus[model_min.status]}")
for i in sedes:
    for j in clientes:
        if rotas[i][j].varValue > 0:
            print(f"Enviar {rotas[i][j].varValue:.0f} cargas de {i} para {j}")
print(f"Custo Total Mínimo: R$ {model_min.objective.value():,.2f}\n")

# ----------------------------------------------------
# CENÁRIO WHAT-IF (Interdição na Rodovia SP-C3 / Alta no Pedágio)
# ----------------------------------------------------
# Custo de SP para C3 salta de R$ 200 para R$ 450 devido ao desvio
custos_crise = {
    'Galpao_SP': {'Cliente_C1': 100, 'Cliente_C2': 150, 'Cliente_C3': 450},
    'Galpao_RJ': {'Cliente_C1': 180, 'Cliente_C2': 120, 'Cliente_C3': 110}
}

model_min_crise = LpProblem(name="Minimização_Crise_Rota", sense=LpMinimize)
rotas_c = LpVariable.dicts("Rota_Crise", (sedes, clientes), lowBound=0, cat="Integer")

model_min_crise += lpSum([rotas_c[i][j] * custos_crise[i][j] for i in sedes for j in clientes])

for i in sedes:
    model_min_crise += lpSum([rotas_c[i][j] for j in clientes]) <= oferta[i]
for j in clientes:
    model_min_crise += lpSum([rotas_c[i][j] for i in sedes]) == demanda[j]

model_min_crise.solve()

print("=== PROBLEMA 2: CENÁRIO WHAT-IF (ROTA ALTERADA) ===")
print(f"Status: {LpStatus[model_min_crise.status]}")
for i in sedes:
    for j in clientes:
        if rotas_c[i][j].varValue > 0:
            print(f"Enviar {rotas_c[i][j].varValue:.0f} cargas de {i} para {j}")
print(f"Custo Ajustado na Crise: R$ {model_min_crise.objective.value():,.2f}")

=== PROBLEMA 2: CENÁRIO BASE ===
Status: Optimal
Enviar 30 cargas de Galpao_SP para Cliente_C1
Enviar 20 cargas de Galpao_SP para Cliente_C2
Enviar 15 cargas de Galpao_RJ para Cliente_C2
Enviar 25 cargas de Galpao_RJ para Cliente_C3
Custo Total Mínimo: R$ 10,550.00

=== PROBLEMA 2: CENÁRIO WHAT-IF (ROTA ALTERADA) ===
Status: Optimal
Enviar 30 cargas de Galpao_SP para Cliente_C1
Enviar 20 cargas de Galpao_SP para Cliente_C2
Enviar 15 cargas de Galpao_RJ para Cliente_C2
Enviar 25 cargas de Galpao_RJ para Cliente_C3
Custo Ajustado na Crise: R$ 10,550.00
